# PSTAT 100 Data Science Concepts and Analysis

## Assignment 3

**Author:**
**Date:**

---

#### Submission Instructions

This assignment will be **due for submission on Wednesday May 27th at 11:59PM**.  Please ensure you submit a `.pdf` file to Canvas by the posted time, no other file types will be accepted.  Please see the course website for some information regarding exporting Jupyter notebooks to pdf, and if you are having exporting problems discuss them with a member of the teaching staff in their office hours.

___

#### Collaboration

You are encouraged to work on your assignments independently.  Please only collaborate with others should you require assistance with more challenging problems. Should you need to collaborate with others, please note their names here:

> **Collaborators:**
___

#### Agent Usage

Additionally, you are permitted to use resources such as ChatGPT and Claude to help you with your homework assignments and to enhance your learning experience. Please make a note of any agents you have used in this submission here:

> **Agents:** 

**WARNING:** Please ensure you check and understand the outputs these agents produce.  Should your solution show evidence of abuse of AI usage (i.e. use external data not detailed in this assignment, use methods not covered in class) your solution will receive a score of 0.

---

## 1. Introduction

In this report we will be performing a statistical analysis to investigate the effect of economic, demographic and health factors on human life expectancy across various countries.  Our study will be utilizing the [Life Expectancy (WHO)](https://www.kaggle.com/datasets/kumarajarshi/life-expectancy-who?resource=download) data, made available on Kaggle by Kumar Rajarshi and originally collected from WHO and United Nations website with the help of Deeksha Russell and Duan Wang.

In this project we consider data from year 2000-2015 for 193 countries for further analysis.  This will involve:

- **Data Preparation:**
    - Data Importing
    - Missingness.
    - Data Transformations.
- **Exploratory Data Analysis:**
    - Summary Statistic Tables
    - Univariate Distributions
    - Bivariate Relationships
- **Modelling:**
    - Fit a Simple Linear Regression (SLR) model. 
    - Fit several Multiple Linear Regression (MLR) models.
    - Evaluate model performance.
    - Residual analysis.

In this report we will be making use of the following libraries:

In [ ]:
# Libraries
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import probplot

# Figure formatting
sns.set_style("whitegrid")
sns.set_palette("Set2") 

---

## 2. Data Preparation

In this section we will load and prepare the data.  Download the `life_expectancy.csv` file from Canvas.  Print the first 5 rows of this data set as well as the shape of the data using the chunk below:

In [ ]:
# Load data
df = pd.read_csv("data/life_expectancy.csv")
# Print shape
print("Shape: ", df.shape)
# Print column names
print(f"  Columns ({len(df.columns)}):")
for c in df.columns:
    print(f"    {c!r}")
# Print first 5 rows
df.head()

We notice that the data column names are inconsistently formatted which will make our life more challenging in future if we do not address it.  To see 

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.1 Column Formatting

Reformat the column names in the following ways:

1. Remove whitespace (spaces before and after) using `strip`.
2. Remove all capitalization.
3. Replace internal spacing with underscores "_".
4. Replace any character that is **not** a lowercase letter, digit or underscore.

**Hint:** For (4) look up what the regex `[^a-z0-9_]` means and how it can be used with `str.replace(...)`.

---

In [ ]:
# Your solution here

---

Use the following chunk to ensure that our column relabelling was successful. You should see 22 column names with only lower case letters, numbers and underscores:

In [ ]:
# Print cleaned column names
print(f"  Columns ({len(df.columns)}):")
for c in df.columns:
    print(f"    {c!r}")

Next we inspect the data types used in each column using the chunk below:

In [ ]:
# Check data types
df.dtypes

All we need to do is convert `status` into a category type which we do as follows:

In [ ]:
# Change status to categorical variable
df["status"] = df["status"].astype("category")

Next we proceed to evaluating the missingness of the data.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.2 Missingness Tables

Produce the following tables:

- A standard missingness table showing the counts and percentages of missing data for each column.
- A missingness table showing the percentages of missing data for each column, grouped by status.

We will be using simple imputation (replacement with median) to handle missing values in numerical columns.  Using this table and by reading the data documentation on Kaggle, comment on the validity of this choice.  Consider the following questions:

- Is the data Missing Completely At Random (MCAR)?
- Might there be underlying patterns to the missingness?

---

In [ ]:
# Your solution here

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2.3 Missing Data Imputation

First, isolate the 20 columns that are numeric and store their names in `numeric_cols`.  Then, write a `for` loop that iterates over these numeric columns and performs the following:

1. Replaces any missing values with the median of that column for that country.
2. Replaces any remaining missing values with the median for that entire column.

Check that your imputation has successfully removed all missing values by using the following:

```
print(f"\n  After imputation — remaining nulls: {df.isnull().sum().sum()}")
```

---

In [ ]:
# Your solution here

---

We conclude our preparation by noting that several covariates are very heavily right skewed.  We can see this by producing a multiplot of the histograms of all numerical variables:

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(16, 15))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col].dropna(), ax=axes[i], bins=30, color="steelblue")
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Count", fontsize=8)

# Hide any unused axes
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Histograms of Numeric Covariates", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

We therefore use the following chunk to take the log of these variables.

In [ ]:
skewed = ['infant_deaths', 'percentage_expenditure', 'measles', 'underfive_deaths', 'hivaids', 'gdp', 'population']

for col in skewed:
    if col in df.columns:
        df[col] = np.log1p(df[col].clip(lower=0))

## 3. Exploratory Data Analysis

### 3.1 Summary Statistics

We proceed to our data exploration and start by producing tables of summary statistics.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.1 Summary Statistics

Produce a table of the key summary statistics (mean, standard deviation, minimum, LQ, median, UQ, max, range, skew and kurtosis) for all numerical covariates.  Provide a few comments on the results including observations about skewed covariates, potential outliers or other points of note.

---

In [ ]:
# Your solution here

---

### 3.2 Univariate Analysis

First, lets consider our categorical covariate `status`. 

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.2 Countplot of `status`

Produce a countplot of the covariate `status` to visualize the proportion of developing countries against developed countries.  Comment on your resulting plot.

---

In [ ]:
# Your solution here

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.3 Histogram Multiplot

Produce a multiplot with 5 rows and 4 columns showing estimates for the distributions for the 20 numerical covariates.  Discuss your findings and the distributions of our transformed covariates.

---

In [ ]:
# Your solution here

---

### 3.3 Bivariate Analysis

We begin our bivariate analysis by looking at the relationship between our response and the categorical variable `status`.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.4 Violin and Grouped Histogram Plot

Produce a multiplot containing the following:

1. A violin plot of `life_expectancy` against `status`; and
2. A histogram of `life_expectancy` grouped by `status`.

Comment on whether you see evidence of a significant difference in the distribution of `life_expectancy` between developed and developing countries.

---

In [ ]:
# Your solution here

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.5 Correlation Heat Map

Produce a correlation heat map of the numerical variables in the dataset.  Write some code to return the covariates with the strongest correlations.  Comment on the strongest positive and negative correlations you observe.  

---

In [ ]:
# Your solution here

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.6 Scatter Multiplots

Produce a multiplot of 4 scatter plots each showing the relationship between `life_expectancy` and the 4 covariates with the highest correlation, with points grouped by `status`.  Note any relationships you see or any strange artifacts of our data preparation / inclusion of multiple years of data.

---

In [ ]:
# Your solution here

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3.7 Life Expectancy Over Time

Make a time series plot of life expectancy over the years grouped by `status`.  Are things moving in the right direction?

---

In [ ]:
# Your solution here

---

## 4. Modelling

### 4.1. Simple Linear Regression

Let's start by fitting a simple linear regression model.  From our exploratory data analysis, we saw that life expectancy is positively correlated with several variables.  We will start with a simple model and then build upon it.

The covariate with the strongest correlation is `hivaids` thus we decide to use this as the sole covariate in our model.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q4.1 SLR

Fit a SLR of `life_expectancy` against `hivaids`.  Print the model summary table and comment on the results. What proportion of the variance in life expectancy is explained by the model?

---

In [ ]:
# Your solution here

---

### 4.2 Multiple Linear Regression

This is an excellent start but lets make things more interesting.  Lets fit a multiple linear regression model using several covariates including a dummy variable for the categorical `status`.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q4.2 MLR 

In this question fit multiple linear regression models with the following covariates:

1. `hivaids`, `gdp`, `underfive_deaths`, `infant_deaths`, `percentage_expenditure`, `measles` and `population`.
2. `hivaids`, `schooling`, `adult_mortality`, `income_composition_of_resources` and `status`.  
3. Your personal selection.

For each, print the model summary table and comment on the results.  Which model performed the best?

---

In [ ]:
# Your solution here

---

## 5. Residual Analysis

Lets consider MLR model 2 from Section 4.2.  We will perform some residual analysis to examine the fit of our model and whether our model assumptions are met.



<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>


#### Q5.1 Residual Analysis

By copying and modifying the code from lecture, produce a multiplot which contains the following 4 residual analysis plots:

1. Residuals vs Fitted
2. Normal Q-Q
3. Scale-Location
4. Cook's Distance

**Note:** The Cook's distance threshold of 4/n is more typically used for smaller data sets and is too strict for large data sets.  Change this to a fixed threshold of 0.5.

Read online about how to read these plots and comment on what each tells us about our model and underlying assumptions.

---

In [ ]:
# Your solution here

---

## Submission

1. Save file to confirm all changes are on disk
2. Run *Kernel > Restart & Run All* to execute all code from top to bottom
3. Save file again to write any new output to disk
4. Export your notebook as a pdf (either through latex or as html before saving as a pdf).
5. Submit your pdf to Canvas.